# An out-of-period test, under a protocol fixed before the run

Chapter 6's held-out test cannot settle whether the final model has skill: the model was chosen after its validation score was seen,
and the test members had been scored during development. This chapter runs the test that is left — the same members' future winters
under SSP3-7.0, a period no choice in this cookbook was made on — under a protocol fixed before the scored run (self-attested; no
external timestamp).


## Overview

1. Why chapter 6's test is not blind, and what a fresh test needs.
2. The protocol, fixed before the scored run, with its decision rule.
3. The future-period transforms, fitted on the 80 training members' future winters only.
4. The frozen models, fitted exactly as chapter 6 fits them.
5. The single scored run: the primary outcome, then the secondary outcomes the protocol lists.
6. The reading against the rule, and the limits.

Shared code lives in `lensml.py` beside the chapters (parameters, transforms, ladder, scoring, paired test, register); `show()` prints a
function's source where a chapter introduces it.


## Prerequisites

| Concepts | Importance | Notes |
|---|---|---|
| [Chapter 6](06-test-and-justify.ipynb) | Necessary | the final model, the test and its caveats |
| [Chapter 2](02-target-and-forced-response.ipynb) | Helpful | the forced response, the wet threshold, the decile bins |
| Distribution shift | Helpful | a model scored on data from another regime than it was fit on |

- **Time to learn**: about 20 minutes; every cell runs in under a minute from the committed caches.
- **System requirements**: the `environment.yml` at the repository root; no account, no local data beyond the committed caches.


## Imports


In [ ]:
import contextlib
import hashlib
import io

import matplotlib.pyplot as plt
import numpy as np
from lensml import (
    FEATURES,
    FUTURE_ACCESS_DATE,
    FUTURE_YEARS,
    RNG_SEED,
    TRIALS,
    apply_transforms,
    at_base_rate,
    cite,
    decile_edges,
    fit_transforms,
    ladder,
    load_future_cache,
    score_frozen,
    setup,
    show,
)

## Why chapter 6's test is not blind

Two things were done with the 20 test members before chapter 6 shows them. During development (2026-10-20) they were scored with two
candidate final models: logistic at the base-rate threshold, which showed no detectable edge, and boosting at the base-rate threshold,
the result chapter 6 prints. And the final model itself was chosen after chapter 3 had already scored it on the training folds. A fresh
test needs data no choice was made on. Every ensemble member has been used — 80 drove every choice, 20 have been scored — so what is
left untouched is the same members' **future** winters: the ssp370 stores of the same ensemble, 2016–2100.


## The protocol, fixed before the scored run

The protocol was written on 2026-10-29, before any SSP3-7.0 data were read, and is followed here without change. That order is
self-attested: there is no external timestamp. The protocol ships verbatim as `data/ssp370-test-protocol.md`; the cell below prints its
SHA-256, so any later change to the file is visible. Its decision rule, in its own words:

> Primary outcome (decided now): precision − base rate for boosting@base-rate on the test members' future winters, with the same
> resampling-over-test-members 95% interval as chapter 6. "Edge replicates" if the interval's lower bound is above 0; "does not
> replicate" otherwise. Secondary, reported whatever they show: AP − base rate with interval; recall and wet-call share; false alarms
> per hit; the same numbers for the comparators; the decile wet rates on the training edges with a chi-square test.

Frozen with it: the model is boosting@base-rate exactly as chapter 6 fits it (all 80 historical training members, the same seed,
hyperparameters and threshold); the comparators are always-wet, always-dry, the Niño-sign rule and logistic@base-rate, fitted the same
way; nothing is refitted on future data. The forced response and the wet threshold for the future period are fitted on the 80 training
members' future winters only, then applied to the test members. One clarification was made before the run, where the protocol left it open:
"the training edges" are chapter 2's decile edges of the training members, the bins chapter 6 used. The scoring code, `score_frozen`, was
checked before the run against chapter 6: on the historical test members it reproduces chapter 6's interval lines exactly.


In [ ]:
show(score_frozen)

In [ ]:
PROTOCOL = 'data/ssp370-test-protocol.md'
with open(PROTOCOL, 'rb') as f:
    sha = hashlib.sha256(f.read()).hexdigest()
print(f'{PROTOCOL} sha256 {sha} (first 16: {sha[:16]})')

## The future table

`data/member_winters_ssp370.csv` was derived with the same `derive_member_winters` as chapter 1 — the same 100 members, boxes, DJF season
and SON autumn — from the ssp370 stores, winters 2016–2100 (DJF 2015 would need December 2014 from the historical store, so it is
dropped). Its header records the experiment, window, parameters and access date, and the reader asserts them. The test members are the
ones chapter 2 named; the cell prints only counts.


In [ ]:
S = (
    setup()
)  # the historical table: training members, test members, chapter 6's model inputs
fut, fut_header = load_future_cache()
print(fut_header)
fut['split'] = np.where(fut['member_id'].isin(S.test_members), 'test', 'train')
for split, rows in fut.groupby('split'):
    print(
        f'{split}: {rows.member_id.nunique()} members · {len(rows)} future winters'
    )

## Future-period transforms, from the training members only

Under SSP3-7.0 the forced path is steeper than in the historical period, so the historical curve cannot be reused: a degree-`FORCED_DEG`
curve and the pooled `WET_QUANTILE` threshold are refitted, on the **training** members' future winters only. The decade check of chapter
2 is repeated on those training members; it reads no test row.


In [ ]:
fut_train = fut[fut['split'].eq('train')]
TF_FUTURE = fit_transforms(fut_train, years=FUTURE_YEARS)
fut = apply_transforms(fut, TF_FUTURE)
fut_train = fut[fut['split'].eq('train')]
print(
    f"future wet threshold (pooled training members) = {TF_FUTURE['wet_threshold']:+.4f} mm/day; "
    f"training-member future base rate {fut_train['wet'].mean():.3f}"
)
decade = (fut_train['winter'] // 10) * 10
for anom in ['nino34_anom', 'prect_anom']:
    g = fut_train.groupby(decade)[anom]
    z = g.mean() / (fut_train[anom].std() / np.sqrt(g.size()))
    print(
        f"{anom:12s} training members, decade-mean anomaly in standard errors: worst {z[z.abs().idxmax()]:+.2f} "
        f"(the {z.abs().idxmax()}s) · tolerance ±3 → {'PASS' if (z.abs() <= 3).all() else 'FAIL'}"
    )

In [ ]:
# The shift in the input, training members only: no test-member future statistic is computed here.
sd_hist = S.train['nino34_anom'].std()
sd_fut = fut_train['nino34_anom'].std()
print(
    f'SON Niño 3.4 anomaly SD, training members: historical {sd_hist:.3f} K · '
    f'future {sd_fut:.3f} K · ratio {sd_fut / sd_hist:.3f}'
)

## The frozen models

Each model is fitted once, on the 80 training members' historical winters, exactly as chapter 6 fits its final model; the final model's
factory is the register's trial 5. Nothing below refits.


In [ ]:
X_tr, y_tr = S.train[FEATURES].values, S.train['wet'].values
name5, make5, _ = TRIALS[4]
assert name5 == 'boosting@base-rate'
models = {
    'boosting@base-rate': make5(RNG_SEED).fit(X_tr, y_tr),
    'always-wet': ladder(RNG_SEED)['always-wet'].fit(X_tr, y_tr),
    'always-dry': ladder(RNG_SEED)['always-dry'].fit(X_tr, y_tr),
    'nino-sign': ladder(RNG_SEED)['nino-sign'].fit(X_tr, y_tr),
    'logistic@base-rate': at_base_rate('logistic')(RNG_SEED).fit(X_tr, y_tr),
}
edges = decile_edges(S.train)  # chapter 2's training edges
print('frozen:', ', '.join(models))

## The scored run

This cell is the test. It prints the primary outcome first, then the secondary outcomes in the protocol's order: for each model, precision
and AP minus the base rate with their intervals, then recall, wet-call share and false alarms per hit; then the decile table with its
chi-square test. A counter marks any re-run in the same kernel; a rebuild recomputes the same numbers and changes nothing.


In [ ]:
SCORED_RUNS = globals().get('SCORED_RUNS', 0) + 1
print(
    f'scored run #{SCORED_RUNS}'
    + ('' if SCORED_RUNS == 1 else '   — a re-run in this kernel')
)
fut_test = fut[fut['split'].eq('test')].reset_index(drop=True)
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    results = score_frozen(models, fut_test, edges)
pt, lo, hi = results['boosting@base-rate']['precision − base rate']
verdict = 'EDGE REPLICATES' if lo > 0 else 'DOES NOT REPLICATE'
print(
    f"PRIMARY OUTCOME — boosting@base-rate, precision − base rate on the test members' future winters: {pt:+.3f}, "
    f'95% interval over test members [{lo:+.3f}, {hi:+.3f}] → {verdict} '
    '(pre-registered rule: replicates only if the lower bound is above 0)'
)
print('\nSECONDARY OUTCOMES, in the protocol\'s order:')
print(buf.getvalue())

In [ ]:
# Chapter 6's estimate, recomputed from the historical test members (already scored there), against this interval.
yhat_hist = models['boosting@base-rate'].predict(S.test[FEATURES].values)
w_hist = S.test['wet'].values
est6 = w_hist[yhat_hist == 1].mean() - w_hist.mean()
print(
    f"chapter 6's estimate (historical test members) {est6:+.3f}; inside this "
    f'interval [{lo:+.3f}, {hi:+.3f}]: {lo <= round(est6, 3) <= hi}'
)

In [ ]:
names = list(results)
fig, ax = plt.subplots(figsize=(7, 3.2))
for i, name in enumerate(names):
    p, a, b = results[name]['precision − base rate']
    ax.plot([a, b], [i, i], color='#8F7022', lw=2)
    ax.plot(
        p, i, 'o', color='#C9A23A', markeredgecolor='#8F7022', markersize=8
    )
ax.axvline(0, color='#13161A', ls='--', lw=1)
ax.set_yticks(range(len(names)), names)
ax.invert_yaxis()
ax.set_xlabel(
    'precision − base rate, test members\' SSP3-7.0 winters (95% interval)'
)
ax.set_title(
    'The scored run: precision against no skill, frozen models', fontsize=9
)
cite(fig, FUTURE_ACCESS_DATE)  # the ssp370 table's access date
plt.tight_layout()

## Reading the run against the rule

**The edge does not replicate under the rule fixed before the run.** The `PRIMARY OUTCOME` line is the result: on the test members'
future winters, boosting at the base-rate threshold has a precision whose 95% interval over test members includes zero. The line after
the secondary outcomes shows the interval also includes chapter 6's estimate, so this test cannot separate no edge from an edge of that
size. It gives no support for leaning an outlook wet.

The secondary outcomes say the same from other sides. AP, which reads the ranking at every threshold, is not distinguishable from no skill
for the final model either (its `AP − base rate` line above). No comparator clears the base rate on precision (the comparator lines printed under the primary outcome): the Niño-sign rule and logistic at the base-rate threshold
sit inside the no-skill band, and always-wet sits on it by construction; always-dry calls nothing wet, so its precision is undefined
(scored 0). Every model that calls wet makes more
false alarms than hits (the `false alarms per hit` values), as precision below 0.5 must. The decile table's chi-square line says whether
the wet rate differs between the training bins on these winters; read it beside chapter 2's and chapter 6's — the coldest bin is the
driest here, as in chapter 2, while in chapter 6's test members the warmest was slightly drier; the omnibus test does not reject chance.

What this does *not* say: it does not show the historical edge was an artefact — a shift in period and forcing could hide a real but
fragile relation (limit 1). It does say that the chapter 6 edge, chosen after its validation score was seen and scored on members already
used, is not evidence a forecaster could lean on.


## Limits

1. **A different period under stronger forcing.** SSP3-7.0 winters are a shifted distribution: the model was fit on 1900–2014 and scored
   on 2016–2100, with its own forced response refitted, and the input itself is wider: the `SON Niño 3.4 anomaly SD` line compares the
   training members' spread in the two periods. A failure can be shift as well as no skill; a success is skill that survived it.
2. **A model world.** CESM2 future winters scored against CESM2 future winters; nothing here touches observations.
3. **The same members.** The 20 test members are the ones scored during development; their future winters are new data, but not new
   realisations of the model's internal variability from scratch.
4. **The existing limits.** Chapter 6's limits 2–7 hold unchanged: CESM2's own ENSO and teleconnection, a ~1° grid, DJF rather than late
   winter–spring, total precipitation, one forced curve for both halves, a box that crosses the Divide.

## Summary

A protocol fixed the model, comparators, outcomes and decision rule before any SSP3-7.0 data were read. The forced response and the wet
threshold for the future period were fitted on the training members' future winters only, and the frozen models were scored once on the
test members' future winters. The primary outcome does not replicate under the rule fixed before the run: the final model's precision
interval includes zero, and also chapter 6's estimate, so the test cannot separate no edge from an edge of that size. It gives no support
for leaning an outlook wet.

### What's next?

The same protocol could be run in another season (FMA), another box or another ensemble, each written down before its data are read.
Observed winters remain the test that would tell a model result from a real one.

## Resources and references

| Resource | Where it bears | Licence |
|---|---|---|
| Grover, M., & Banihirwe, A. (2021). *NCAR CESM2 Large Ensemble Cloud-Optimized Dataset*. NSF NCAR CISL. https://doi.org/10.26024/y48t-q717 | the ssp370 stores (required citation) | UCAR Data Terms of Use |
| Rodgers, K. B. et al. (2021). *Earth System Dynamics* 12, 1393–1411. https://doi.org/10.5194/esd-12-1393-2021 | the ensemble and its SSP3-7.0 forcing (required citation) | CC BY 4.0 |
| scikit-learn User Guide, §12 Common pitfalls. https://scikit-learn.org/stable/common_pitfalls.html | transforms fit on training data only | BSD-3-Clause |

Results based on the CESM2 Large Ensemble acknowledge the CESM2 Large Ensemble Community Project and supercomputing resources provided by
the IBS Center for Climate Physics in South Korea.
